In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
from PIL import Image, ImageDraw, ImageFont

session = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])
labels = {0: 'title', 1: 'plain text', 2: 'abandon', 3: 'figure', 4: 'figure_caption', 5: 'table', 6: 'table_caption', 7: 'table_footnote', 8: 'isolate_formula', 9: 'formula_caption'}
model_h, model_w = 1024, 1024


def preprocess(img_paths):
    if isinstance(img_paths, str):
        img_paths = [img_paths]

    batch = []
    for img_path in img_paths:
        img = cv2.imread(img_path)
        img_resized = cv2.resize(img, (model_w, model_h))
        img_input = img_resized[:, :, ::-1].transpose(2, 0, 1)
        img_input = img_input.astype(np.float32) / 255.0
        batch.append(img_input)

    batch = np.stack(batch, axis=0)
    return {session.get_inputs()[0].name: batch}



def nms(boxes, scores, iou_threshold=0.5):
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        inter = w * h
        iou = inter / (areas[i] + areas[order[1:]] - inter)
        inds = np.where(iou <= iou_threshold)[0]
        order = order[inds + 1]
    return np.array(keep, dtype=np.int64)


In [49]:
img_files = [f"pages/{i}.png" for i in range(1, 9)]
images = preprocess(img_files)
outputs = session.run(None, images)

batch_preds = outputs[0]  # shape (B, 300, 6)

font = ImageFont.truetype("arial.ttf", 48)
results = []

for img_file, preds in zip(img_files, batch_preds):
    boxes = preds[:, :4]
    scores = preds[:, 4]
    classes = preds[:, 5].astype(int)

    keep = nms(boxes, scores, iou_threshold=0.5)
    boxes, scores, classes = boxes[keep], scores[keep], classes[keep]

    img = Image.open(img_file).convert("RGB")
    x_factor, y_factor = img.width / model_w, img.height / model_h
    draw = ImageDraw.Draw(img)

    for box, score, cls in zip(boxes, scores, classes):
        if score < 0.1:
            continue
        x1, y1, x2, y2 = box
        x1, y1, x2, y2 = (
            int(x1 * x_factor),
            int(y1 * y_factor),
            int(x2 * x_factor),
            int(y2 * y_factor),
        )
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, y1), f"{labels[cls]}:{score:.2f}", fill="red", font=font)

    img.show()


(8, 3, 1024, 1024)


In [42]:
outputs[0].shape

(8, 300, 6)